# GEM-Bench Evaluation Analysis

Qualitative and Quantitative evaluation of ad-injected response quality across different **Base LLM** and **Judge LLM** combinations.

In [32]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

OUTPUT_BASE = '/home/gpuuser7/gpuuser7_a/prateek/GEM-Bench/GemBench/benchmarking/output/evaluate_results/output'

# Experiment configurations
EXPERIMENTS = [
    {
        'base_llm': 'llama-3.3-70b',
        'judge': 'qwen-32b',
        'dir': '20260917_001526_judge_qwen32b_base_llama70b',
    },
    {
        'base_llm': 'llama-3.3-70b',
        'judge': 'deepseek-r1-distill-qwen-32b',
        'dir': '20260917_063902_judge_deepseek_r1_base_llama70b',
    },
    {
        'base_llm': 'qwen-3-30b-a3b',
        'judge': 'llama-3.3-70b',
        'dir': '20260921_102040_judge_llama-3.3-70b-instruct-fp8_base_qwen3-30b-a3b',
    },
    {
        'base_llm': 'deepseek-r1-distill-qwen-32b',
        'judge': 'llama-3.3-70b',
        'dir': '20260921_234312_judge_llama-3.3-70b-instruct-fp8_base_deepseek-r1-distill-qwen-32b',
    },
    {
        'base_llm': 'gemma-3-27b-it',
        'judge': 'llama-3.3-70b',
        'dir': '20260922_131025_judge_llama-3.3-70b-instruct-fp8_base_gemma-3-27b-it',
    },
    {
        'base_llm': 'gemma-3-27b-it',
        'judge': 'deepseek-r1-distill-qwen-32b',
        'dir': '20260922_183041_judge_deepseek-r1-distill-qwen-32b_base_gemma-3-27b-it',
    },
    {
        'base_llm': 'gemma-3-27b-it',
        'judge': 'qwen-32b',
        'dir': '20260923_072301_judge_qwen3-32b_base_gemma-3-27b-it',
    },
]

QUALITATIVE_METRICS = ['accuracy', 'naturalness', 'personality', 'trust', 'notice_products', 'click_products']
METRIC_DISPLAY = {
    'accuracy': 'Accuracy',
    'naturalness': 'Naturalness',
    'personality': 'Personality',
    'trust': 'Trust',
    'notice_products': 'Notice',
    'click_products': 'Click',
}

QUANTITATIVE_METRICS = ['local measure', 'global_coherence', 'global measure', 'ad_content_alignment', 'has_ad', 'product_selection_accuracy']
QUANT_METRIC_DISPLAY = {
    'local measure': 'RF',
    'global_coherence': 'RC',
    'global measure': 'AF',
    'ad_content_alignment': 'AC',
    'has_ad': 'IR',
    'product_selection_accuracy': 'CTR',
}


In [33]:
def load_results(xlsx_path):
    """Parse the evaluation_result_average.xlsx format."""
    df = pd.read_excel(xlsx_path, header=None)
    
    # Build column names from rows 1 (top-level) and 2 (metric names)
    row1 = df.iloc[1].tolist()
    row2 = df.iloc[2].tolist()
    cols = []
    for r1, r2 in zip(row1, row2):
        if pd.notna(r1) and str(r1) != 'nan':
            cols.append(str(r1))
        elif pd.notna(r2) and str(r2) != 'nan':
            cols.append(str(r2))
        else:
            cols.append('unknown')
    
    data = df.iloc[3:].reset_index(drop=True)
    data.columns = cols
    data['data_set'] = data['data_set'].ffill()
    
    # Convert metric columns to numeric
    for m in QUALITATIVE_METRICS:
        if m in data.columns:
            data[m] = pd.to_numeric(data[m], errors='coerce')
    for m in QUANTITATIVE_METRICS:
        if m in data.columns:
            data[m] = pd.to_numeric(data[m], errors='coerce')
    
    return data

# Load all experiments
all_results = []
for exp in EXPERIMENTS:
    path = f"{OUTPUT_BASE}/{exp['dir']}/evaluation_result_average.xlsx"
    df = load_results(path)
    df['base_llm'] = exp['base_llm']
    df['judge'] = exp['judge']
    all_results.append(df)

combined = pd.concat(all_results, ignore_index=True)
print(f"Loaded {len(combined)} rows from {len(EXPERIMENTS)} experiments")
print(f"Datasets: {combined['data_set'].unique().tolist()}")
print(f"Solutions: {combined['solution'].unique().tolist()}")
print(f"Base LLMs: {combined['base_llm'].unique().tolist()}")
print(f"Judges: {combined['judge'].unique().tolist()}")


Loaded 84 rows from 7 experiments
Datasets: ['CA_Prod', 'LM-Market', 'MT-Human']
Solutions: ['Ad-Chat', 'GI-R', 'GIR-R', 'GIR-P']
Base LLMs: ['llama-3.3-70b', 'qwen-3-30b-a3b', 'deepseek-r1-distill-qwen-32b', 'gemma-3-27b-it']
Judges: ['qwen-32b', 'deepseek-r1-distill-qwen-32b', 'llama-3.3-70b']


In [34]:
def build_paper_table(combined_df, dataset_filter=None, solution_filter=None):
    """Build a table in the paper format: Dataset | Base LLM | Judge LLM | Solution | Metrics."""
    
    df = combined_df.copy()
    if dataset_filter:
        df = df[df['data_set'].isin(dataset_filter)]
    if solution_filter:
        df = df[df['solution'].isin(solution_filter)]
    
    # Build the table rows
    rows = []
    for ds in df['data_set'].unique():
        ds_data = df[df['data_set'] == ds]
        for base in ds_data['base_llm'].unique():
            base_data = ds_data[ds_data['base_llm'] == base]
            for judge in base_data['judge'].unique():
                judge_data = base_data[base_data['judge'] == judge]
                for sol in judge_data['solution'].unique():
                    sol_data = judge_data[judge_data['solution'] == sol]
                    row = {
                        'Dataset': ds,
                        'Base LLM': base,
                        'Judge LLM': judge,
                        'Solution': sol,
                    }
                    vals_for_overall = []
                    for m in QUALITATIVE_METRICS:
                        val = sol_data[m].values[0]
                        if pd.notna(val) and str(val).lower() != 'nan':
                            row[METRIC_DISPLAY[m]] = round(float(val), 2)
                            vals_for_overall.append(float(val))
                        else:
                            row[METRIC_DISPLAY[m]] = '—'
                    
                    if len(vals_for_overall) > 0:
                        row['Overall'] = round(sum(vals_for_overall) / len(vals_for_overall), 2)
                    else:
                        row['Overall'] = '—'
                    rows.append(row)
    
    table = pd.DataFrame(rows)
    return table

# Build full table with Ad-Chat and GIR-R (matching paper format)


In [35]:
def style_table(df, title=''):
    """Style a DataFrame to bold the best value per (Dataset, Solution) group for each metric."""
    metric_cols = list(METRIC_DISPLAY.values()) + ['Overall']
    
    def bold_max_by_dataset(data):
        styles = pd.DataFrame('', index=data.index, columns=data.columns)
        for ds in data['Dataset'].unique():
            ds_idx = data[data['Dataset'] == ds].index
            for col in metric_cols:
                if col in data.columns:
                    numeric_s = pd.to_numeric(data.loc[ds_idx, col], errors='coerce')
                    if numeric_s.notna().any():
                        max_val = numeric_s.max()
                        is_max = numeric_s == max_val
                        styles.loc[ds_idx[is_max], col] = 'font-weight: bold'
        return styles
    
    styled = (df.style
              .apply(bold_max_by_dataset, axis=None)
              .set_caption(title)
              .format(precision=2)
              .set_table_styles([
                  {'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold'), ('padding', '10px')]},
                  {'selector': 'th', 'props': [('background-color', '#f0f0f0'), ('padding', '8px')]},
                  {'selector': 'td', 'props': [('padding', '6px 12px')]},
              ])
              .hide(axis='index'))
    return styled


In [36]:
def style_quant_table(df, title=''):
    metric_cols = list(QUANT_METRIC_DISPLAY.values()) + ['Overall']
    
    def bold_max_by_dataset(data):
        styles = pd.DataFrame('', index=data.index, columns=data.columns)
        for ds in data['Dataset'].unique():
            ds_idx = data[data['Dataset'] == ds].index
            for col in metric_cols:
                if col in data.columns:
                    numeric_s = pd.to_numeric(data.loc[ds_idx, col], errors='coerce')
                    if numeric_s.notna().any():
                        max_val = numeric_s.max()
                        is_max = numeric_s == max_val
                        styles.loc[ds_idx[is_max], col] = 'font-weight: bold'
        return styles
    
    styled = (df.style
              .apply(bold_max_by_dataset, axis=None)
              .set_caption(title)
              .format(precision=2)
              .set_table_styles([
                  {'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold'), ('padding', '10px')]},
                  {'selector': 'th', 'props': [('background-color', '#f0f0f0'), ('padding', '8px')]},
                  {'selector': 'td', 'props': [('padding', '6px 12px')]},
              ])
              .hide(axis='index'))
    return styled


## Results by Base LLM and Judge LLM

Displays the Qualitative and Quantitative metric tables for each unique combination of Base LLM and Judge LLM.

In [37]:
combinations = combined[['base_llm', 'judge']].drop_duplicates().values
for base, judge in combinations:
    subset = combined[(combined['base_llm'] == base) & (combined['judge'] == judge)]
    if len(subset) > 0:
        print(f"\n{'='*60}\nBase: {base} | Judge: {judge}\n{'='*60}")
        
        # Qualitative
        qual_table = build_paper_table(subset)
        display(style_table(qual_table, title=f'Qualitative Metrics (Base: {base}, Judge: {judge})'))
        
        # Quantitative
        quant_table = build_quant_paper_table(subset)
        display(style_quant_table(quant_table, title=f'Quantitative Metrics (Base: {base}, Judge: {judge})'))



Base: llama-3.3-70b | Judge: qwen-32b


Dataset,Base LLM,Judge LLM,Solution,Accuracy,Naturalness,Personality,Trust,Notice,Click,Overall
CA_Prod,llama-3.3-70b,qwen-32b,Ad-Chat,39.46,34.21,30.66,24.17,18.26,76.12,37.15
CA_Prod,llama-3.3-70b,qwen-32b,GI-R,42.75,27.67,33.33,27.33,13.50,70.75,35.89
CA_Prod,llama-3.3-70b,qwen-32b,GIR-R,53.67,37.92,37.50,32.92,22.50,75.08,43.26
CA_Prod,llama-3.3-70b,qwen-32b,GIR-P,52.25,37.58,37.58,32.75,25.08,75.83,43.51
LM-Market,llama-3.3-70b,qwen-32b,Ad-Chat,68.66,52.37,56.08,50.32,42.96,58.55,54.82
LM-Market,llama-3.3-70b,qwen-32b,GI-R,70.75,45.05,64.78,48.71,38.60,64.35,55.38
LM-Market,llama-3.3-70b,qwen-32b,GIR-R,74.46,52.96,61.61,52.20,48.55,67.20,59.50
LM-Market,llama-3.3-70b,qwen-32b,GIR-P,74.14,50.97,61.51,52.37,50.86,67.10,59.49
MT-Human,llama-3.3-70b,qwen-32b,Ad-Chat,74.00,51.00,53.00,57.00,32.00,48.00,52.50
MT-Human,llama-3.3-70b,qwen-32b,GI-R,78.00,46.00,71.00,52.00,37.00,65.00,58.17


Dataset,Base LLM,Judge LLM,Solution,RF,RC,AF,AC,IR,CTR,Overall
CA_Prod,llama-3.3-70b,qwen-32b,Ad-Chat,36.85,64.97,82.68,55.29,100.00,41.05,63.47
CA_Prod,llama-3.3-70b,qwen-32b,GI-R,37.22,67.64,83.17,61.05,100.00,31.67,63.46
CA_Prod,llama-3.3-70b,qwen-32b,GIR-R,51.75,75.60,84.99,61.84,100.00,31.67,67.64
CA_Prod,llama-3.3-70b,qwen-32b,GIR-P,51.86,75.75,85.00,61.04,100.00,35.00,68.11
LM-Market,llama-3.3-70b,qwen-32b,Ad-Chat,39.59,63.02,83.24,49.89,78.85,—,62.92
LM-Market,llama-3.3-70b,qwen-32b,GI-R,40.66,65.88,82.82,52.10,100.00,—,68.29
LM-Market,llama-3.3-70b,qwen-32b,GIR-R,40.26,64.80,80.82,54.19,100.00,—,68.01
LM-Market,llama-3.3-70b,qwen-32b,GIR-P,39.87,64.50,81.61,51.91,100.00,—,67.58
MT-Human,llama-3.3-70b,qwen-32b,Ad-Chat,42.95,63.35,83.37,43.82,76.67,—,62.03
MT-Human,llama-3.3-70b,qwen-32b,GI-R,41.66,66.88,88.73,40.87,100.00,—,67.63



Base: llama-3.3-70b | Judge: deepseek-r1-distill-qwen-32b


Dataset,Base LLM,Judge LLM,Solution,Accuracy,Naturalness,Personality,Trust,Notice,Click,Overall
CA_Prod,llama-3.3-70b,deepseek-r1-distill-qwen-32b,Ad-Chat,50.08,49.26,30.91,28.43,19.17,76.20,42.34
CA_Prod,llama-3.3-70b,deepseek-r1-distill-qwen-32b,GI-R,60.17,40.33,37.67,30.17,22.58,67.50,43.07
CA_Prod,llama-3.3-70b,deepseek-r1-distill-qwen-32b,GIR-R,63.50,56.33,45.17,31.42,25.92,69.75,48.68
CA_Prod,llama-3.3-70b,deepseek-r1-distill-qwen-32b,GIR-P,64.42,56.50,44.25,31.33,28.42,69.92,49.14
LM-Market,llama-3.3-70b,deepseek-r1-distill-qwen-32b,Ad-Chat,71.08,58.60,57.90,45.70,38.66,55.48,54.57
LM-Market,llama-3.3-70b,deepseek-r1-distill-qwen-32b,GI-R,76.94,51.94,65.91,45.32,40.81,58.98,56.65
LM-Market,llama-3.3-70b,deepseek-r1-distill-qwen-32b,GIR-R,79.25,59.95,63.98,48.33,43.66,58.60,58.96
LM-Market,llama-3.3-70b,deepseek-r1-distill-qwen-32b,GIR-P,80.22,60.11,62.96,48.44,41.99,61.24,59.16
MT-Human,llama-3.3-70b,deepseek-r1-distill-qwen-32b,Ad-Chat,75.00,57.00,63.00,56.00,29.00,47.00,54.50
MT-Human,llama-3.3-70b,deepseek-r1-distill-qwen-32b,GI-R,84.00,52.00,77.00,52.00,30.00,56.00,58.50


...

Dataset,Base LLM,Judge LLM,Solution,Accuracy,Naturalness,Personality,Trust,Notice,Click,Overall
CA_Prod,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,Ad-Chat,59.13,59.67,50.37,31.49,37.52,74.13,52.05
CA_Prod,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GI-R,62.46,43.32,42.54,31.72,20.74,68.03,44.80
CA_Prod,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GIR-R,71.15,60.45,52.05,34.43,25.82,68.81,52.12
CA_Prod,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GIR-P,72.19,61.74,53.22,34.46,28.26,68.72,53.10
LM-Market,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,Ad-Chat,84.46,63.44,60.48,46.51,58.98,70.81,64.11
LM-Market,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GI-R,87.63,60.11,77.96,59.35,45.86,61.34,65.38
LM-Market,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GIR-R,87.96,65.27,70.91,58.23,48.06,64.03,65.74
LM-Market,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GIR-P,87.58,63.87,69.25,57.42,49.25,63.98,65.22
MT-Human,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,Ad-Chat,87.00,68.00,67.00,55.00,50.00,59.00,64.33
MT-Human,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GI-R,89.00,62.00,82.00,70.00,45.00,61.00,68.17


Dataset,Base LLM,Judge LLM,Solution,RF,RC,AF,AC,IR,CTR,Overall
CA_Prod,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,Ad-Chat,27.88,52.25,79.32,37.21,100.00,41.87,56.42
CA_Prod,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GI-R,37.52,67.31,83.94,60.61,100.00,32.24,63.60
CA_Prod,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GIR-R,42.82,70.86,82.72,56.85,100.00,32.24,64.25
CA_Prod,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GIR-P,42.74,71.01,82.69,56.26,100.00,36.09,64.80
LM-Market,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,Ad-Chat,30.56,52.56,82.96,46.96,98.92,—,62.39
LM-Market,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GI-R,35.51,57.92,79.94,48.98,100.00,—,64.47
LM-Market,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GIR-R,32.96,55.13,75.53,48.28,100.00,—,62.38
LM-Market,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GIR-P,32.89,55.05,75.10,46.24,100.00,—,61.86
MT-Human,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,Ad-Chat,32.61,51.05,86.70,38.05,80.00,—,57.68
MT-Human,gemma-3-27b-it,deepseek-r1-distill-qwen-32b,GI-R,37.05,58.82,81.04,39.97,100.00,—,63.38



Base: gemma-3-27b-it | Judge: qwen-32b


Dataset,Base LLM,Judge LLM,Solution,Accuracy,Naturalness,Personality,Trust,Notice,Click,Overall
CA_Prod,gemma-3-27b-it,qwen-32b,Ad-Chat,42.52,44.13,39.88,30.25,32.44,75.83,44.17
CA_Prod,gemma-3-27b-it,qwen-32b,GI-R,47.17,28.44,36.56,27.70,11.31,67.75,36.49
CA_Prod,gemma-3-27b-it,qwen-32b,GIR-R,63.28,40.57,44.80,34.34,21.80,74.14,46.49
CA_Prod,gemma-3-27b-it,qwen-32b,GIR-P,62.93,41.90,45.54,34.88,25.54,74.38,47.53
LM-Market,gemma-3-27b-it,qwen-32b,Ad-Chat,79.30,52.04,55.43,46.99,55.22,77.26,61.04
LM-Market,gemma-3-27b-it,qwen-32b,GI-R,80.81,48.76,71.77,52.74,45.43,68.82,61.39
LM-Market,gemma-3-27b-it,qwen-32b,GIR-R,82.10,53.39,66.77,56.29,48.01,71.99,63.09
LM-Market,gemma-3-27b-it,qwen-32b,GIR-P,81.77,50.59,66.08,53.71,50.05,70.32,62.09
MT-Human,gemma-3-27b-it,qwen-32b,Ad-Chat,87.00,54.00,57.00,51.00,40.00,60.00,58.17
MT-Human,gemma-3-27b-it,qwen-32b,GI-R,87.00,51.00,74.00,62.00,37.00,66.00,62.83


Dataset,Base LLM,Judge LLM,Solution,RF,RC,AF,AC,IR,CTR,Overall
CA_Prod,gemma-3-27b-it,qwen-32b,Ad-Chat,27.88,52.25,79.32,37.21,100.00,41.87,56.42
CA_Prod,gemma-3-27b-it,qwen-32b,GI-R,37.52,67.31,83.94,60.61,100.00,32.24,63.60
CA_Prod,gemma-3-27b-it,qwen-32b,GIR-R,42.82,70.86,82.72,56.85,100.00,32.24,64.25
CA_Prod,gemma-3-27b-it,qwen-32b,GIR-P,42.74,71.01,82.69,56.26,100.00,36.09,64.80
LM-Market,gemma-3-27b-it,qwen-32b,Ad-Chat,30.56,52.56,82.96,46.96,98.92,—,62.39
LM-Market,gemma-3-27b-it,qwen-32b,GI-R,35.51,57.92,79.94,48.98,100.00,—,64.47
LM-Market,gemma-3-27b-it,qwen-32b,GIR-R,32.96,55.13,75.53,48.28,100.00,—,62.38
LM-Market,gemma-3-27b-it,qwen-32b,GIR-P,32.89,55.05,75.10,46.24,100.00,—,61.86
MT-Human,gemma-3-27b-it,qwen-32b,Ad-Chat,32.61,51.05,86.70,38.05,80.00,—,57.68
MT-Human,gemma-3-27b-it,qwen-32b,GI-R,37.05,58.82,81.04,39.97,100.00,—,63.38
